In [ ]:
import os
import math
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score
from PIL import Image
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
!pip install timm -q

In [ ]:
TRAIN_DIR = r'/kaggle/input/competitions/cse-281-spring-26-scene-style-classification/StyleClassificationIndoors/StyleClassificationIndoors/train'
print('Train exists:', os.path.exists(TRAIN_DIR))

# Remove corrupted images
def remove_bad_images(root):
    removed = 0
    for folder, _, files in os.walk(root):
        for file in files:
            path = os.path.join(folder, file)
            try:
                with Image.open(path) as img:
                    img.verify()
            except Exception:
                os.remove(path)
                removed += 1
    print(f'Removed {removed} bad images')

remove_bad_images(TRAIN_DIR)

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandAugment(num_ops=2, magnitude=9), # 🔥 THE REGULARIZATION BEAST
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std =[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2),
])

# THE FIX: No more squashed aspect ratios for validation!
val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

In [ ]:
full_train = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
full_val   = datasets.ImageFolder(root=TRAIN_DIR, transform=val_transforms)
print('Total images:', len(full_train))

targets = full_train.targets
indices = list(range(len(targets)))
train_idx, val_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=targets
)
print(f'Train: {len(train_idx)} | Val: {len(val_idx)}')

train_dataset = Subset(full_train, train_idx)
val_dataset   = Subset(full_val,   val_idx)

# Weighted sampler for class imbalance
train_targets  = [targets[i] for i in train_idx]
class_counts   = np.bincount(train_targets)
class_weights  = 1.0 / class_counts
sample_weights = [class_weights[t] for t in train_targets]
sampler = WeightedRandomSampler(
    weights=torch.FloatTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

# Dropped to 16 to fit the massive 86M parameter model in VRAM
train_loader = DataLoader(train_dataset, batch_size=16, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
import torch
import torch.nn as nn

class DINOv2Classifier(nn.Module):
    def __init__(self, num_classes=17):
        super().__init__()
        # 🔥 Upgraded to the Base (86M) model!
        self.backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.5), 
            nn.Linear(768, num_classes) # 🔥 Must be upgraded from 384 to 768
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

# Initialize your new beast
model = DINOv2Classifier(num_classes=17)
model = model.to(device)

# Freeze the backbone completely to start
for param in model.backbone.parameters():
    param.requires_grad = False

# Only unfreeze the classifier head for the warmup
for param in model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,}')

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

In [ ]:
NUM_EPOCHS = 40 

# 1. INITIAL SETUP: Unfreeze ONLY the classifier head
for name, param in model.named_parameters():
    if 'classifier' in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# 2. CREATE THE STARTER OPTIMIZER (Conservative Head LR + Heavy Weight Decay)
# optimizer = torch.optim.AdamW(
#     filter(lambda p: p.requires_grad, model.parameters()),
#     lr=5e-4, weight_decay=1e-2 
# )
# 2. CREATE THE STARTER OPTIMIZER (SGD with Momentum)
optimizer = torch.optim.SGD(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-2,            # 🔥 20x higher than Adam's LR
    momentum=0.9,       # 🔥 Crucial for SGD!
    weight_decay=1e-4,  # Standard SGD weight decay
    nesterov=True       # Adds a smarter momentum calculation
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-5)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

# 3. DINOv2 DYNAMIC UNFREEZE TRACKERS
# We unfreeze in chunks from the back of the transformer
blocks = model.backbone.blocks
layers_to_unfreeze = [
    [model.backbone.norm, blocks[10], blocks[11]], # Group 1: Norm + Last 2 blocks
    [blocks[8], blocks[9]],                        # Group 2: Blocks 8 & 9
    [blocks[6], blocks[7]],                        # Group 3: Blocks 6 & 7
    [blocks[4], blocks[5]]                         # Group 4: Blocks 4 & 5
]

unfreeze_index = 0
plateau_counter = 0
patience = 2 # Dropped to 4 so it unfreezes before massive overfitting sets in

# 🔥 TRACKING VAL LOSS NOW, NOT ACCURACY 🔥
best_val_acc = 0.0
best_val_loss = float('inf') 

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train BAcc':>10} | {'Val Loss':>8} | {'Val BAcc':>8} | {'Best':>6}")
print('-' * 65)

# Add this near the top of Cell 4
ACCUMULATION_STEPS = 4 # 4 steps of batch size 16 = Effective Batch Size of 64

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    
    # Notice we moved zero_grad outside the loop!
    optimizer.zero_grad() 
    
    # Add enumerate so we can track the batch index (i)
    for i, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        
        out  = model(images)
        loss = criterion(out, labels)
        
        # Scale the loss down because we are adding it up 4 times
        loss = loss / ACCUMULATION_STEPS 
        loss.backward()
        
        # Only step the optimizer every 4th batch (Gradient Accumulation!)
        if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad() # Reset for the next accumulation cycle
            
        # Multiply back by accumulation steps just for the print logs
        total_loss += loss.item() * ACCUMULATION_STEPS 
        all_preds.extend(out.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
    return total_loss / len(loader), balanced_accuracy_score(all_labels, all_preds)

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            out  = model(images)
            loss = criterion(out, labels)
            total_loss += loss.item()
            all_preds.extend(out.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), balanced_accuracy_score(all_labels, all_preds)

for epoch in range(NUM_EPOCHS):
    
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss,   val_acc   = validate(model, val_loader, criterion, device)

    scheduler.step()

# --- DUAL-CONDITION VALIDATION LOGIC ---
    is_best = False
    
    # Condition 1: We hit a brand new highest accuracy! 
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_val_loss = val_loss # Save the loss that achieved this accuracy
        is_best = True
        
    # Condition 2: Accuracy is tied, but the Loss dropped (The model became more confident!)
    elif val_acc == best_val_acc and val_loss < best_val_loss:
        best_val_loss = val_loss
        is_best = True

    if is_best:
        plateau_counter = 0 # Reset plateau because we genuinely improved!
        torch.save(model.state_dict(), '/kaggle/working/best_dinov2_dynamic.pth')
    else:
        plateau_counter += 1 # Increment if neither condition was met
    # ---------------------------------------

    print(f"{epoch+1:>5} | {train_loss:>10.4f} | {train_acc:>10.4f} | {val_loss:>8.4f} | {val_acc:>8.4f} | {'✓' if is_best else f'Plateau: {plateau_counter}/{patience}'}")

    # --- DYNAMIC UNFREEZE TRIGGER ---
    if plateau_counter >= patience and unfreeze_index < len(layers_to_unfreeze):
        print(f"\n🔥 Plateau Hit! Dynamically unfreezing Transformer Block Group {unfreeze_index + 1}... 🔥")
        
        # Unfreeze the target list of modules
        for module in layers_to_unfreeze[unfreeze_index]:
            for param in module.parameters():
                param.requires_grad = True
            
        unfreeze_index += 1
        plateau_counter = 0 # Reset counter after unfreezing
        
        # Re-initialize Optimizer for the newly unfrozen parameters
        # print("   -> Resetting optimizer with safe ViT learning rates.")
        # optimizer = torch.optim.AdamW([
        #     {'params': model.classifier.parameters(), 'lr': 5e-4},
        #     # Apply a tiny 1e-5 learning rate to ANY backbone parameter that is currently unfrozen
        #     {'params': [p for n, p in model.named_parameters() if 'classifier' not in n and p.requires_grad], 'lr': 1e-5}
        # ], weight_decay=1e-2)
        # Re-initialize Optimizer for the newly unfrozen parameters
        print("   -> Resetting SGD optimizer with differential ViT learning rates.")
        optimizer = torch.optim.SGD([
            {'params': model.classifier.parameters(), 'lr': 1e-2}, # Head stays fast
            # Unfrozen backbone layers get a slower LR so they don't break
            {'params': [p for n, p in model.named_parameters() if 'classifier' not in n and p.requires_grad], 'lr': 1e-3}
        ], momentum=0.9, weight_decay=1e-4, nesterov=True)
        
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-5)
        
        # scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)
    # -------------------------------------

print(f'\nBest Val Loss Achieved: {best_val_loss:.4f}')

         _______
         \_____/
          \o|o/     / 
           \_/     /  
       ----|ᵒ|----/
      /    |ᵒ|    
     /     ___    ᵒPIZZAᵒᵒᵒSTEVEᵒ 
           / \
          /   \
         /     \
▔▔▔▔▔▔▔▔▔▔▔▔▔▔▔▔

In [ ]:
import os
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ==========================================
# 1. LOAD THE .PTH MODEL
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# IMPORTANT: You must define the "empty shell" of your model architecture first
# Example: model = torchvision.models.resnet18(num_classes=17)

# Load the weights from your .pth file
pth_file_path = "/kaggle/working/best_dinov2_dynamic.pth" # <--- REPLACE WITH YOUR PTH FILE PATH
model.load_state_dict(torch.load(pth_file_path, map_location=device))

model.to(device)
model.eval() # IMPORTANT: Turns off the Dropout constraint for maximum testing power

# ==========================================
# 2. DATASET & DATALOADER SETUP
# ==========================================
class TestDataset(Dataset):
    def __init__(self, folder_path, transform=None):
        self.folder_path = folder_path
        self.image_files = [f for f in os.listdir(folder_path) if f.endswith(('.jpg', '.jpeg', '.png'))]
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.folder_path, img_name)
        
        try:
            # Try to open the image normally
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            # IF IT FAILS: Print a warning and create a fake black image to prevent a crash
            print(f"  [!] Warning: {img_name} is corrupted. Using a blank placeholder.")
            image = Image.new('RGB', (224, 224), (0, 0, 0))
        
        if self.transform:
            image = self.transform(image)
            
        return image, img_name
    
# Setting up the testing data
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_folder_path = r'/kaggle/input/competitions/cse-281-spring-26-scene-style-classification/StyleClassificationIndoors/StyleClassificationIndoors/test'

test_dataset = TestDataset(folder_path=test_folder_path, transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

# ==========================================
# 3. RUN INFERENCE
# ==========================================
predictions = []

print(f"Starting inference on {len(test_dataset)} test images...")

with torch.no_grad():
    # Added enumerate(test_loader) here so your 'i' variable works for the print statement
    for i, (images, img_names) in enumerate(test_loader): 
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        for img_name, pred in zip(img_names, predicted):
            predictions.append({'ImageName': img_name, 'ClassLabel': pred.item()})
            
        if (i+1) % 25 == 0:
            print(f"  Processed batch {i+1}/{len(test_loader)}...")

# ==========================================
# 4. FORMAT AND SAVE CSV
# ==========================================
submission_df = pd.DataFrame(predictions)

# ---> THE SORTING UPGRADE <---
# 1. Extract the number from the string using a Regular Expression and convert to integer
submission_df['sort_number'] = submission_df['ImageName'].str.extract(r'(\d+)').astype(int)

# 2. Sort the DataFrame using our new integer column
submission_df = submission_df.sort_values(by='sort_number')

# 3. Drop the temporary sorting column so it doesn't end up in the final CSV
submission_df = submission_df.drop(columns=['sort_number'])

# Save to disk
submission_df.to_csv('Eyad_model_submission_V7.csv', index=False)
print("SUCCESS! Output saved to 'Eyad_model_submission_V8.csv' in perfect numerical order.")